In [3]:
from google.cloud import bigquery
from pandas_gbq import to_gbq
import pandas as pd
import requests
import re
import csv
import os
import warnings
warnings.filterwarnings('ignore')

In [4]:
def get_json_data(api_url):
    headers = {
        'User-Agent': 'scu cchen22@scu.edu'
    }
    # 'User-Agent' Sample: Company Name AdminContact@<sample company domain>.com
    # "Accept-Encoding": "gzip, deflate",
    # "Host": "data.sec.gov"
    # api doc: https://www.sec.gov/edgar/sec-api-documentation

    response = requests.get(api_url, headers=headers)
    if response.status_code == 200:
        json_data = response.json()
        return json_data
    else:
        print("Error:", response.status_code)
        return None

In [5]:
# Load the yield dictionary
project_id = 'silicon-data-421520'
client = bigquery.Client(project=project_id)
query = """
SELECT *
FROM `AssetBenchmarkDB.YieldDict`
"""

yield_dict = client.query(query).to_dataframe()

In [6]:
# Generate Yield Table
df_yield_concat = pd.DataFrame()
# Get API data: Cols 2~5 of yield dict list the tags for the required concept (Cash & CE, ST, LT, Interest Income)
yield_columns = yield_dict.columns[2:6]
for index, row in yield_dict.iterrows():
    df_yield = pd.DataFrame()
    cik = str(row['company_CIK']).strip().zfill(10)
    for column in yield_columns:
        if pd.isna(row[column]):
            df_yield[column] = 0
            continue
        split_data_names = re.split(r'([+\-])', row[column])
        df_tag = pd.DataFrame()
        # Support summing multiple values. (e.g.Fortinet: ST = Short-term investments + Marketable equity securities)
        operator = "+"
        for item in split_data_names:
            if item == "+" or item == "-":
                operator = item
            else:
                tag = item
                json_data = get_json_data(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{tag}.json")
                df_tag_i = pd.DataFrame(json_data['units']['USD'])
                df_tag_i = df_tag_i[(df_tag_i['frame'].notna()) & (df_tag_i['form'].isin(['10-Q', '10-K']))]
                df_tag_i['val'] = (df_tag_i['val'] / 1000000).astype(int)
                df_tag_i.drop(columns=['accn', 'fy', 'fp', 'filed', 'form', 'frame'], inplace=True)
                df_tag_i.rename(columns={'val': item}, inplace=True)
                if df_tag.empty:
                    df_tag = df_tag_i.rename(columns={item: column})
                else:
                    df_tag = pd.merge(df_tag, df_tag_i, on=['end'], how='left')
                    df_tag.fillna(0, inplace=True)
                    if operator == "+":
                        df_tag[column] += df_tag[item]
                        df_tag.drop(columns=item, inplace=True)
                    elif operator == "-":
                        df_tag[column] -= df_tag[item]
                        df_tag.drop(columns=item, inplace=True)
        # Merge each unique concept data into the yield table
        if df_yield.empty:
            df_yield = df_tag
        else:
            df_yield = pd.merge(df_yield, df_tag, on=['end'], how='left')
            df_yield.fillna(0, inplace=True)
    df_yield.insert(0, 'company_CIK', row['company_CIK'])
    df_yield.insert(0, 'company_name', row['company_name'])
    df_yield.rename(columns={'end': 'end_date', 'start': 'interest_start_date'}, inplace=True)
    df_yield['end_date'] = pd.to_datetime(df_yield['end_date'])
    df_yield['interest_start_date'] = pd.to_datetime(df_yield['interest_start_date'], errors='coerce')
    df_yield.sort_values(by="end_date", inplace=True)
    # Calculate quarterly interest income. (Minus the three quarters value from the previous 3 rows if the interest income is annual)
    df_yield['interest_income_quarterly'] = 0.0
    for index_comp, row_comp in df_yield.iterrows():
        date_diff = (row_comp['end_date'] - row_comp['interest_start_date']).days
        if abs(date_diff - 90) <= 10:
            df_yield.at[index_comp, 'interest_income_quarterly'] = row_comp['interest_income']
        elif abs(date_diff - 365) <= 50 and index_comp >= 3:
            df_yield.at[index_comp, 'interest_income_quarterly'] = row_comp['interest_income']
            for i in (1, 2, 3):
                if df_yield.at[index_comp - i, 'end_date'] <= row_comp['interest_start_date'] \
                        or (row_comp['end_date'] - df_yield.at[index_comp - i, 'interest_start_date']).days > 400 \
                        or pd.isna(df_yield.at[index_comp - i, 'interest_start_date']):
                    df_yield.at[index_comp, 'interest_income_quarterly'] /= 4
                    break
                else:
                    df_yield.at[index_comp, 'interest_income_quarterly'] -= df_yield.at[
                        index_comp - i, 'interest_income']
    df_yield['total_cash'] = df_yield['cash_and_CE'] + df_yield['short_term_investments'] + df_yield[
        'long_term_investments']
    df_yield = df_yield[df_yield['total_cash'] > 0]
    df_yield['average_total_cash'] = 0.0
    df_yield['yield'] = 0.0
    # Calculate the average total cash for the most recent 2 quarters
    for index_comp, row_comp in df_yield.iloc[1:].iterrows():
        df_yield.at[index_comp, 'average_total_cash'] = (row_comp['total_cash'] + df_yield.at[
            index_comp - 1, 'total_cash']) / 2
        df_yield.at[index_comp, 'yield'] = (
                    row_comp['interest_income_quarterly'] / df_yield.at[index_comp, 'average_total_cash'] * 365 / 90 * 100).round(2)
    df_yield['end_date_minus_45'] = df_yield['end_date'] - pd.DateOffset(days=45)
    # Use the date difference between the start and end time to determine 10-Q or 10-K
    df_yield['form'] = '10-Q'
    date_diff = (df_yield['end_date'] - df_yield['interest_start_date']).dt.days
    df_yield.loc[(date_diff >= 150) & (date_diff < 400), 'form'] = '10-K'
    df_yield = df_yield[df_yield['end_date'] >= '2015']
    df_yield_concat = pd.concat([df_yield_concat, df_yield], ignore_index=True, join='outer')

In [7]:
# Load the maturity dictionary
project_id = 'silicon-data-421520'
client = bigquery.Client(project=project_id)
query = """
SELECT *
FROM `AssetBenchmarkDB.MaturityDict`
"""

maturity_dict = client.query(query).to_dataframe()
maturity_dict['maturity_tag'] = maturity_dict['maturity_tag'].apply(eval)

In [8]:
df_maturity_concat = pd.DataFrame()
for index, row in maturity_dict.iterrows():
    df_maturity = pd.DataFrame()
    cik = str(row['company_CIK']).strip().zfill(10)
    for i, tag in row["maturity_tag"].items():
        json_data = get_json_data(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{tag}.json")
        tag_data = json_data['units']['USD']
        df_tag_data = pd.DataFrame(tag_data)
        df_tag_data = df_tag_data[df_tag_data['end'] >= "2015"]
        df_tag_data = df_tag_data[df_tag_data['frame'].notna() & df_tag_data['form'].isin(['10-Q', '10-K'])]
        df_tag_data['val'] = (df_tag_data['val'] / 1000000).astype(int)
        df_tag_data = df_tag_data.drop(columns=['accn', 'fy', 'fp', 'filed', 'frame']).rename(
            columns={'end': "end_date", 'val': i})
        df_tag_data['end_date'] = pd.to_datetime(df_tag_data['end_date'])
        if df_maturity.empty:
            df_maturity = df_tag_data
        else:
            df_maturity = pd.merge(df_maturity, df_tag_data, on=['end_date', 'form'], how='outer')
    df_maturity = df_maturity.fillna(0)
    cols = list(df_maturity.columns)
    cols.insert(0, cols.pop(cols.index('form')))
    df_maturity = df_maturity[cols]
    # Handle an API error manually: for apple, one of the valuefrom 10-K was marked as 10-Q in the api
    if row['company_name'] == 'Apple':
        apple_drop_row = (df_maturity['form'] == '10-Q') & (df_maturity['end_date'] == '2023-09-30')
        apple_keep_row = (df_maturity['form'] == '10-K') & (df_maturity['end_date'] == '2023-09-30')
        df_maturity.loc[apple_keep_row, '0.5'] = df_maturity.loc[apple_drop_row, '0.5'].values
        df_maturity = df_maturity[~apple_drop_row ].reset_index(drop=True)
    df_maturity.insert(0, 'company_CIK', row['company_CIK'])
    df_maturity.insert(0, 'company_name', row['company_name'])
    maturity_start_index = df_maturity.columns.get_loc("end_date") + 1
    df_maturity["total"] = df_maturity.iloc[:, maturity_start_index:].sum(axis=1)
    maturity_end_index = df_maturity.columns.get_loc("total")
    df_maturity["maturity"] = 0.0
    for i in range(maturity_start_index, maturity_end_index):
        df_maturity["maturity"] += float(df_maturity.columns[i]) * df_maturity.iloc[:, i]
    df_maturity["maturity"] = (df_maturity["maturity"] / df_maturity["total"]).round(2)
    df_maturity['end_date_minus_45'] = df_maturity['end_date'] - pd.DateOffset(days=45)
    df_maturity = df_maturity[df_maturity['end_date'] >= '2015']
    df_maturity_concat = pd.concat([df_maturity_concat, df_maturity], ignore_index=True, join='outer')

df_maturity_concat.columns = df_maturity_concat.columns.str.replace('.', '_')
columns_reordered = [col for col in df_maturity_concat.columns if col not in ['total', 'maturity', 'end_date_minus_45']] + ['total', 'maturity', 'end_date_minus_45']
df_maturity_concat = df_maturity_concat[columns_reordered]

In [9]:
df_combine = pd.merge(df_yield_concat, df_maturity_concat, on=['company_CIK', 'end_date'], how='inner')
df_combine.columns = [col[:-2] if col.endswith('_x') else col for col in df_combine.columns]
df_combine = df_combine[[col for col in df_combine.columns if not col.endswith('_y')]]

In [10]:
type(df_combine)

pandas.core.frame.DataFrame

In [11]:
# #METHOD 1 - Import data to the GCP Database without using a Service Account in the code
table_id = 'AssetBenchmarkDB.YieldTable'
to_gbq(df_yield_concat, table_id, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 5729.92it/s]


In [12]:
table_id = 'AssetBenchmarkDB.MaturityTable'
to_gbq(df_maturity_concat, table_id, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 8439.24it/s]


In [13]:
table_id = 'AssetBenchmarkDB.CombinedTable'
to_gbq(df_combine, table_id, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 8525.01it/s]


In [14]:
#METHOD 2 - TO LOAD DATAFRAME(DATA) TO BIG QUERY TABLE USING PYTHON CLIENT LIBRARY
# import datetime
# from google.cloud import bigquery
# import pandas
# import pytz
# from google.oauth2 import service_account

In [15]:
# def load_table_dataframe_config(key_path,project_id,table_id, data):
#     credentials = service_account.Credentials.from_service_account_file(
#         key_path, scopes=["https://www.googleapis.com/auth/cloud-platform"],
#         )

#     # Construct a BigQuery client object.
#     client = bigquery.Client(credentials=credentials, project=project_id)

#     job_config = bigquery.LoadJobConfig(
#         write_disposition="WRITE_TRUNCATE")

#     job = client.load_table_from_dataframe(
#         data, table_id, job_config=job_config
#     )
#     job.result()

#     data = client.get_table(table_id)
#     return data

In [16]:
##key_path is the path to service account json file
# key_path= "/content/silicon-data-421520-ca0b00d4ceaa.json"
# project_id="silicon-data-421520"
# dataset_id = "AssetBenchmarkDB"


In [17]:
# table="YieldTable"
# table_id="{}.{}.{}".format(project_id, dataset_id, table)
# print("********* NAME OF TABLE IS", table_id)

# dataYield = load_table_dataframe_config(key_path,project_id, table_id, df_yield_concat)

In [18]:

# table="MaturityTable"
# table_id="{}.{}.{}".format(project_id, dataset_id, table)
# print("********* NAME OF TABLE IS", table_id)

# dataMaturity = load_table_dataframe_config(key_path,project_id, table_id, df_maturity_concat)

In [19]:
# table="CombinedTable"
# table_id="{}.{}.{}".format(project_id, dataset_id, table)
# print("********* NAME OF TABLE IS", table_id)

# dataCombined = load_table_dataframe_config(key_path,project_id, table_id, df_combine)